# SMART GROW - Entrenamiento de Modelos de Machine Learning

Este notebook contiene el proceso completo de entrenamiento de los modelos predictivos:
1. **Modelo de Heladas** (XGBoost) - Predicción 12-24 horas
2. **Modelo de Enfermedades** (Random Forest) - Condiciones favorables para hongos

**Autor:** Cristian Rodríguez  
**Proyecto:** SMART GROW - UTN-FRT 2025

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import math
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Configuración
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Generación de Dataset Sintético

Generamos datos sintéticos calibrados para el clima de Tucumán, Argentina.

In [ ]:
def generate_synthetic_data(n_samples=52993):
    """
    Genera dataset sintético calibrado para Tucumán.
    
    Parámetros climáticos de referencia:
    - Temperatura media anual: 19.3°C
    - Humedad relativa media: 70%
    - Presión media: 1013 hPa
    - Período de heladas: Mayo-Agosto
    """
    
    # Generar timestamps (1 año de datos cada 10 minutos)
    start_date = datetime(2024, 1, 1)
    timestamps = [start_date + timedelta(minutes=10*i) for i in range(n_samples)]
    
    data = []
    
    for ts in timestamps:
        month = ts.month
        hour = ts.hour
        
        # Temperatura base según mes (ciclo anual)
        if month in [12, 1, 2]:  # Verano
            temp_base = 28 + np.random.normal(0, 3)
        elif month in [3, 4, 5]:  # Otoño
            temp_base = 20 + np.random.normal(0, 4)
        elif month in [6, 7, 8]:  # Invierno
            temp_base = 12 + np.random.normal(0, 5)
        else:  # Primavera
            temp_base = 22 + np.random.normal(0, 4)
        
        # Variación diurna
        if 6 <= hour <= 18:
            temp_mod = 5 * math.sin((hour - 6) * math.pi / 12)
        else:
            temp_mod = -3
        
        temperature = temp_base + temp_mod
        
        # Humedad (inversa a temperatura)
        humidity = 90 - (temperature - 10) * 1.5 + np.random.normal(0, 8)
        humidity = np.clip(humidity, 30, 98)
        
        # Temperatura del suelo (más estable)
        soil_temp = temperature * 0.7 + 5 + np.random.normal(0, 1)
        
        # Presión barométrica
        pressure = 1013 + np.random.normal(0, 8)
        
        # Cambio de presión (simulado)
        pressure_change = np.random.normal(0, 3)
        
        # Luz (según hora)
        if 6 <= hour <= 18:
            light = 50000 * math.sin((hour - 6) * math.pi / 12) + np.random.normal(0, 5000)
            light = max(0, light)
        else:
            light = np.random.uniform(0, 10)
        
        # Humedad del suelo
        soil_humidity = 50 + np.random.normal(0, 15)
        soil_humidity = np.clip(soil_humidity, 10, 95)
        
        # Punto de rocío
        a, b = 17.27, 237.7
        alpha = ((a * temperature) / (b + temperature)) + math.log(humidity / 100.0)
        dew_point = (b * alpha) / (a - alpha)
        
        # Diferencial térmico
        temp_differential = temperature - soil_temp
        
        # ETIQUETA DE HELADA
        # Condiciones: temp < 2°C O (temp < 5°C AND dew_point < 0 AND pressure > 1015)
        frost = 0
        if temperature < 2:
            frost = 1
        elif temperature < 5 and dew_point < 0 and pressure > 1015 and light < 100:
            frost = 1 if np.random.random() < 0.7 else 0
        elif temperature < 8 and dew_point < 2 and pressure_change > 5:
            frost = 1 if np.random.random() < 0.3 else 0
        
        # ETIQUETA DE ENFERMEDAD
        # Condiciones: humedad > 85% por >6h AND 15 < temp < 25
        disease_favorable = 0
        if humidity > 85 and 15 <= temperature <= 25:
            disease_favorable = 1 if np.random.random() < 0.8 else 0
        elif humidity > 80 and 18 <= temperature <= 22:
            disease_favorable = 1 if np.random.random() < 0.5 else 0
        
        data.append({
            'timestamp': ts,
            'temperature': round(temperature, 2),
            'humidity': round(humidity, 2),
            'soil_temp': round(soil_temp, 2),
            'soil_humidity': round(soil_humidity, 2),
            'pressure': round(pressure, 2),
            'pressure_change': round(pressure_change, 2),
            'light': round(light, 2),
            'dew_point': round(dew_point, 2),
            'temp_differential': round(temp_differential, 2),
            'hour': hour,
            'month': month,
            'frost': frost,
            'disease_favorable': disease_favorable
        })
    
    return pd.DataFrame(data)

# Generar datos
df = generate_synthetic_data()
print(f"Dataset generado: {len(df)} registros")
print(f"\nDistribución de heladas: {df['frost'].value_counts().to_dict()}")
print(f"Distribución de enfermedades: {df['disease_favorable'].value_counts().to_dict()}")
df.head()

## 2. Modelo de Predicción de Heladas (XGBoost)

In [ ]:
# Features para modelo de heladas
frost_features = [
    'temperature', 'humidity', 'soil_temp', 'temp_differential',
    'dew_point', 'pressure', 'pressure_change', 'light', 'hour', 'month'
]

X_frost = df[frost_features]
y_frost = df['frost']

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_frost, y_frost, test_size=0.2, random_state=42, stratify=y_frost
)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print(f"Proporción heladas en train: {y_train.mean():.2%}")

In [ ]:
# Entrenar XGBoost
frost_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=len(y_train[y_train==0]) / len(y_train[y_train==1]),  # Balance de clases
    random_state=42,
    eval_metric='logloss'
)

frost_model.fit(X_train, y_train)

# Predicciones
y_pred = frost_model.predict(X_test)
y_prob = frost_model.predict_proba(X_test)[:, 1]

# Métricas
print("=" * 50)
print("MODELO DE HELADAS (XGBoost)")
print("=" * 50)
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.2%}")
print(f"Precision: {precision_score(y_test, y_pred):.2%}")
print(f"Recall:    {recall_score(y_test, y_pred):.2%}")
print(f"F1-Score:  {f1_score(y_test, y_pred):.2%}")
print(f"AUC-ROC:   {roc_auc_score(y_test, y_prob):.2%}")
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred, target_names=['No Helada', 'Helada']))

In [ ]:
# Importancia de features
importance = pd.DataFrame({
    'feature': frost_features,
    'importance': frost_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance, x='importance', y='feature', palette='viridis')
plt.title('Importancia de Variables - Modelo de Heladas')
plt.xlabel('Importancia')
plt.tight_layout()
plt.savefig('../models/frost_feature_importance.png', dpi=150)
plt.show()

print(importance)

## 3. Modelo de Predicción de Enfermedades (Random Forest)

In [ ]:
# Features para modelo de enfermedades
disease_features = [
    'temperature', 'humidity', 'soil_humidity', 'light', 'hour', 'month'
]

X_disease = df[disease_features]
y_disease = df['disease_favorable']

# Split
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_disease, y_disease, test_size=0.2, random_state=42, stratify=y_disease
)

# Entrenar Random Forest
disease_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    class_weight='balanced',
    random_state=42
)

disease_model.fit(X_train_d, y_train_d)

# Predicciones
y_pred_d = disease_model.predict(X_test_d)
y_prob_d = disease_model.predict_proba(X_test_d)[:, 1]

# Métricas
print("=" * 50)
print("MODELO DE ENFERMEDADES (Random Forest)")
print("=" * 50)
print(f"Accuracy:  {accuracy_score(y_test_d, y_pred_d):.2%}")
print(f"Precision: {precision_score(y_test_d, y_pred_d):.2%}")
print(f"Recall:    {recall_score(y_test_d, y_pred_d):.2%}")
print(f"F1-Score:  {f1_score(y_test_d, y_pred_d):.2%}")
print(f"AUC-ROC:   {roc_auc_score(y_test_d, y_prob_d):.2%}")

## 4. Guardar Modelos Entrenados

In [ ]:
# Guardar modelos
joblib.dump(frost_model, '../models/frost_model.joblib')
joblib.dump(disease_model, '../models/disease_model.joblib')

print("✓ Modelos guardados en ../models/")
print("  - frost_model.joblib")
print("  - disease_model.joblib")

## 5. Validación Cruzada

In [ ]:
# Cross-validation para modelo de heladas
cv_scores = cross_val_score(frost_model, X_frost, y_frost, cv=5, scoring='f1')
print(f"Cross-Validation F1 (Heladas): {cv_scores.mean():.2%} (+/- {cv_scores.std()*2:.2%})")

cv_scores_d = cross_val_score(disease_model, X_disease, y_disease, cv=5, scoring='f1')
print(f"Cross-Validation F1 (Enfermedades): {cv_scores_d.mean():.2%} (+/- {cv_scores_d.std()*2:.2%})")

## Notas Importantes

⚠️ **Limitaciones del Dataset Sintético:**

1. Los datos fueron generados con distribuciones aproximadas al clima de Tucumán
2. No captura eventos extremos reales ni correlaciones complejas
3. La validación en campo es **INDISPENSABLE** antes del uso en producción

📋 **Próximos pasos:**

1. Recolectar datos reales durante 6-12 meses
2. Validar predicciones contra eventos de helada documentados
3. Reentrenar modelos con datos reales
4. Ajustar umbrales según contexto local